# PNNL Richland WA — Environmental Conditions, May–June 2023

This notebook compiles two months of environmental forcing data for the Pacific Northwest  
National Laboratory (PNNL) Richland field site, drawing on three complementary remote-sensing  
and reanalysis products.

| Source | Variables | Grid |
|---|---|---|
| NASA POWER MERRA-2 | Temperature (mean/min/max), precipitation, relative humidity | 0.5° (~55 km) |
| NASA POWER SYN1deg | SW ↓, LW ↓, PAR (all-sky & clear-sky) | 1° (~111 km) |
| Sentinel-5P TROPOMI | CO total column (single overpass) | ~5.5 × 3.5 km |

The field site (~40 m × 30 m) is well within a single grid cell for all products;  
NASA POWER queries use the bbox centroid.


In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd

from env_data_mcp.sources.nasa_power import (
    TemporalResolution,
    nasa_power_merra2_query,
    nasa_power_syn1deg_query,
)
from env_data_mcp.sources.sentinel5p import sentinel5p_bbox_query

PNNL_BBOX = {
    "min_lat": 46.251407,
    "max_lat": 46.251790,
    "min_lon": -119.728785,
    "max_lon": -119.728369,
}
PNNL_START = "2023-05-01"
PNNL_END = "2023-06-30"  # full two months

CENTROID_LAT = (PNNL_BBOX["min_lat"] + PNNL_BBOX["max_lat"]) / 2
CENTROID_LON = (PNNL_BBOX["min_lon"] + PNNL_BBOX["max_lon"]) / 2

S5P_DATE = "2023-05-15"  # single overpass for CO column

## 1. Fetch Site Data

### NASA POWER — MERRA-2 meteorology and SYN1deg surface radiation


In [ ]:
merra2_r = nasa_power_merra2_query(
    latitude=CENTROID_LAT,
    longitude=CENTROID_LON,
    start_date=PNNL_START,
    end_date=PNNL_END,
    temporal_resolution=TemporalResolution.DAILY,
)
df_merra2 = pd.DataFrame(merra2_r["data"])
df_merra2["date"] = pd.to_datetime(df_merra2["date"])

syn1deg_r = nasa_power_syn1deg_query(
    latitude=CENTROID_LAT,
    longitude=CENTROID_LON,
    start_date=PNNL_START,
    end_date=PNNL_END,
    temporal_resolution=TemporalResolution.DAILY,
)
df_syn1deg = pd.DataFrame(syn1deg_r["data"])
df_syn1deg["date"] = pd.to_datetime(df_syn1deg["date"])

### Sentinel-5P TROPOMI — CO total column

A single overpass on 2023-05-15 is queried to place atmospheric CO in context with the  
meteorological and radiation record. Each S5P granule is ~150 MB; expect 30–120 s.


In [ ]:
s5p_r = sentinel5p_bbox_query(
    min_lat=PNNL_BBOX["min_lat"],
    max_lat=PNNL_BBOX["max_lat"],
    min_lon=PNNL_BBOX["min_lon"],
    max_lon=PNNL_BBOX["max_lon"],
    start_date=S5P_DATE,
    end_date=S5P_DATE,
    product="CO",
)

# Extract CO value for annotation on the radiation figure (None if no overpass).
s5p_co = s5p_r["data"][0].get("CO_mean") if s5p_r["data"] else None

## 2. Analysis

### Meteorological Overview — May–June 2023


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

# --- Temperature ---
axes[0].fill_between(
    df_merra2["date"],
    df_merra2["T2M_MIN"],
    df_merra2["T2M_MAX"],
    alpha=0.2,
    color="#d62728",
    label="Daily min–max range",
)
axes[0].plot(
    df_merra2["date"], df_merra2["T2M"], color="#d62728", linewidth=1.5, label="Daily mean T (2 m)"
)
axes[0].set_ylabel("Temperature (°C)")
axes[0].legend(fontsize=9, loc="upper left")
axes[0].grid(alpha=0.25, linewidth=0.5)

# --- Precipitation ---
axes[1].bar(df_merra2["date"], df_merra2["PRECTOTCORR"], color="#1f77b4", alpha=0.75, width=0.9)
axes[1].set_ylabel("Precipitation\n(mm day⁻¹)")
axes[1].grid(alpha=0.25, axis="y", linewidth=0.5)

# --- Relative Humidity ---
axes[2].plot(df_merra2["date"], df_merra2["RH2M"], color="#2ca02c", linewidth=1.2)
axes[2].set_ylabel("Relative Humidity\nat 2 m (%)")
axes[2].set_ylim(0, 100)
axes[2].grid(alpha=0.25, linewidth=0.5)
axes[2].xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
axes[2].xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0))
plt.setp(axes[2].xaxis.get_majorticklabels(), rotation=30, ha="right")

for ax in axes:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig.suptitle(
    f"PNNL Richland WA — Meteorological Conditions  ({PNNL_START} to {PNNL_END})\n"
    "MERRA-2 reanalysis  ·  0.5° grid",
    fontsize=11,
)
plt.tight_layout()
plt.show()

### Surface Radiation Overview — May–June 2023

The dotted vertical line marks the Sentinel-5P overpass date (2023-05-15).  
The shaded region in the PAR panel shows the **cloud reduction** (clear-sky minus all-sky).


In [ ]:
s5p_dt = pd.to_datetime(S5P_DATE)

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

# --- Shortwave radiation ---
axes[0].plot(
    df_syn1deg["date"],
    df_syn1deg["ALLSKY_SFC_SW_DWN"],
    color="#ff7f0e",
    linewidth=1.5,
    label="All-sky SW ↓",
)
axes[0].set_ylabel("Shortwave Radiation\n(W m⁻²)")
axes[0].legend(fontsize=9, loc="upper left")
axes[0].grid(alpha=0.25, linewidth=0.5)

# --- PAR: all-sky vs clear-sky, cloud reduction shaded ---
axes[1].fill_between(
    df_syn1deg["date"],
    df_syn1deg["ALLSKY_SFC_PAR_TOT"],
    df_syn1deg["CLRSKY_SFC_PAR_TOT"],
    alpha=0.3,
    color="#9467bd",
    label="Cloud reduction",
)
axes[1].plot(
    df_syn1deg["date"],
    df_syn1deg["CLRSKY_SFC_PAR_TOT"],
    color="#9467bd",
    linestyle="--",
    linewidth=1.0,
    alpha=0.7,
    label="Clear-sky PAR",
)
axes[1].plot(
    df_syn1deg["date"],
    df_syn1deg["ALLSKY_SFC_PAR_TOT"],
    color="#9467bd",
    linewidth=1.5,
    label="All-sky PAR",
)
axes[1].set_ylabel("PAR\n(W m⁻²)")
axes[1].legend(fontsize=9, loc="upper left")
axes[1].grid(alpha=0.25, linewidth=0.5)
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
axes[1].xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0))
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=30, ha="right")

# Mark Sentinel-5P overpass and annotate with CO value
co_label = f"S5P CO = {s5p_co:.3f} mol m⁻²" if s5p_co is not None else "S5P overpass"
for ax in axes:
    ax.axvline(s5p_dt, color="gray", linestyle=":", linewidth=1.2, zorder=3)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
axes[0].annotate(
    co_label,
    xy=(s5p_dt, df_syn1deg["ALLSKY_SFC_SW_DWN"].max() * 0.95),
    xytext=(10, 0),
    textcoords="offset points",
    fontsize=8,
    color="gray",
)

fig.suptitle(
    f"PNNL Richland WA — Surface Radiation  ({PNNL_START} to {PNNL_END})\n"
    "CERES SYN1deg  ·  1° grid",
    fontsize=11,
)
plt.tight_layout()
plt.show()